# PROTMIND — Inference & Testing Notebook
Notebook ini digunakan untuk menguji secara interaktif alur inferensi deteksi kepatuhan APD (Stage 1 & Stage 2) baik untuk Laptop (development) maupun Jetson Nano (produksi).

In [ ]:
import os
import sys
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Masukkan path proyek ke sys.path
project_dir = Path(os.getcwd())
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src.deployment.main_jetson import load_yolo_model, estimate_distance, CONFIG, play_audio
print("Library & modul berhasil dimuat!")
print("CUDA Available:", torch.cuda.is_available())

### 1. Inisialisasi Model APD Dinamis
Selanjutnya, kita akan memuat model Person Detector dan PPE Detector. Secara otomatis sistem akan memprioritaskan model `.engine` TensorRT jika CUDA tersedia, atau beralih ke `.pt`/`.onnx` secara aman.

In [ ]:
print("Memuat model...")
model_person = load_yolo_model(CONFIG["model_person"], {0: "person"}, is_person=True)
model_ppe = load_yolo_model(CONFIG["model_ppe"], {
    0: "helmet",
    1: "no-helmet",
    2: "no-safety shoes",
    3: "no-vest",
    4: "safety shoes",
    5: "vest"
}, is_person=False)
print("Model Person:", CONFIG["model_person"])
print("Model PPE:", CONFIG["model_ppe"])

### 2. Uji Coba Audio Speaker
Jalankan cell di bawah untuk mengetes apakah speaker USB RS260 Anda sudah bersuara.

In [ ]:
test_audio_file = CONFIG["audio_helm"]
print("Memutar audio tes:", test_audio_file)
play_audio(test_audio_file)

### 3. Uji Coba Deteksi pada Gambar Tunggal
Mari kita jalankan inferensi 2-tahap pada satu gambar uji coba dan tampilkan hasilnya secara visual.

In [ ]:
# Tentukan path gambar Anda (ubah jika perlu)
img_path = "yolov8n.pt"

# Gunakan dummy frame jika file gambar tidak ditemukan
frame = np.zeros((480, 640, 3), dtype=np.uint8)
cv2.putText(frame, "PROTMIND TEST FRAME", (150, 240), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

# Jika ada gambar nyata, baca gambar tersebut
# frame = cv2.imread("path_ke_gambar_anda.jpg")

H, W, _ = frame.shape
annotated_frame = frame.copy()

# Jalankan Stage 1: Person Detection
results_p = model_person(frame, classes=0, conf=CONFIG["conf_person"], verbose=False)
person_boxes = results_p[0].boxes.xyxy.cpu().numpy() if hasattr(results_p[0].boxes, 'xyxy') else results_p[0].boxes

print(f"Terdeteksi {len(person_boxes)} orang.")

for p_box in person_boxes:
    px1, py1, px2, py2 = map(int, p_box[:4])
    cv2.rectangle(annotated_frame, (px1, py1), (px2, py2), (0, 255, 0), 2)
    
    # Crop & Detect APD
    person_crop = frame[py1:py2, px1:px2]
    if person_crop.size > 0:
        results_ppe = model_ppe(person_crop, conf=CONFIG["conf_ppe"], verbose=False)

# Tampilkan frame
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()